## Spectrum Plotter

This tool takes a folder of JETI spectrophotometer CSV exports and turns them
into comparison plots.

**What it does, step by step:**

1. **Load** — point it at a folder; it reads every CSV inside and pulls out all
   the individual radiance measurements. Fresh-leaf and pre-burial ("initcond")
   samples are automatically excluded, since those are reference points rather
   than decomposition data.
2. **Group** — pick which observations (e.g. specific soil × leaf × time-point
   combinations) should appear together on the same plot. You can build as many
   groups/plots as you want in one run.
3. **Customize** — for each group, give it a title, and give each observation
   inside it a legend label and a colour.
4. **Set the wavelength window** — choose the nm range shown on the x-axis
   (applies to all groups).
5. **Choose plot type** — raw radiance, AUC-normalized (each spectrum divided
   by its own area under the curve, so shape can be compared independent of
   intensity), or both.

**Output:** PNG plots saved into `Spectrum_plots_RAW/` and/or
`Spectrum_plots_AUC/` subfolders inside your data directory, one file per plot
group, named after that group's title and the wavelength range used.

The tool also quietly handles messy/inconsistent formatting in the JETI export
files (decimal separators, duplicate column names) so you don't have to clean
the CSVs by hand first.

In [ ]:
"""
Spectrum Plotter
================

A small Tkinter application for exploring JETI spectrophotometer CSV
exports.

Workflow
--------
1. Pick a folder containing spectrum CSV files.
2. Choose what to do with the loaded data: save everything into one
   combined table (CSV, TXT and/or Excel), create plots, or both.
3. If plotting, build one or more "plot groups": each group is a set of
   observations that will be drawn together. Every group gets its own
   title (shown on the figure or only used in the file name), plus a
   custom legend label and colour per observation.
4. Choose a wavelength window shared by all groups.
5. Choose which plot types to create (raw radiance and the available
   normalization methods).
6. Choose the image formats (PNG, JPEG, SVG, PDF, TIFF, AVIF) and the
   resolution (DPI) of the saved figures.

Input format
------------
A JETI export holds, for several spectra side by side (one CSV column
per spectrum):

* a header with metadata (device, name, date, integration time, ...),
* scalar results (luminance, CCT, chromaticity, ...),
* one or more spectral blocks, each starting with a
  ``Wavelength [nm],...`` line (calibrated radiance, raw counts, ...).

Historically, JETI writes every decimal number as two CSV fields
(``1234,56`` instead of ``1234.56``), which is what you get when a tool
saves a CSV under a locale that uses ``,`` as the decimal separator.
Rather than trusting a global layout, every row is decoded on its own:
with N spectra, a row carrying N fields per row uses one field per value
and a row carrying 2N fields uses the split layout. The parser therefore
keeps working if JETI fixes their export.

Nothing in the file is discarded: every header field, scalar and spectral
block is kept and written to the combined table in its own column.
"""

from __future__ import annotations

import csv
import functools
import importlib.util
import io
import logging
import re
import tkinter as tk
from collections.abc import Callable, Iterable, Iterator, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from tkinter import colorchooser, filedialog, messagebox, ttk

import matplotlib as mpl
import numpy as np
import pandas as pd
from matplotlib import font_manager
from matplotlib.figure import Figure

# Root at WARNING keeps third-party libraries quiet (fontTools logs dozens of
# INFO lines per PDF); this application's own logger reports at INFO.
logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")
log = logging.getLogger("spectrum_plotter")
log.setLevel(logging.INFO)


# =====================================================================
# Configuration
# =====================================================================

APP_TITLE = "Spectrum Plotter"
BUTTON_COLUMNS = 3

COMBINED_FILENAME_STEM = "combined_corrected_spectra"
EXCEL_MAX_ROWS = 1_048_576

DEFAULT_DPI = 300
MIN_DPI, MAX_DPI = 50, 1200
FIGURE_SIZE_INCHES = (10, 6)

# Labels used by JETI exports.
NAME_LABEL = "Name"
WAVELENGTH_LABEL = "Wavelength [nm]"
RADIANCE_BLOCK_KEY = "Le [W/(sr*sqm*nm)]"  # the block that gets plotted
INTEGRATION_TIME_LABEL = "T_int [ms]"      # header field, e.g. "14 * 500.000"

COLOR_PALETTE = [
    "#004D40",  # Dark Teal
    "#E69F00",  # Amber Orange
    "#56B4E9",  # Sky Blue
    "#009E73",  # Bluish Green
    "#F0E442",  # Warm Yellow
    "#0072B2",  # Royal Blue
    "#D55E00",  # Vermilion
    "#CC79A7",  # Reddish Purple
]

# Figure typography: Times New Roman, then metric-compatible or similar
# serif fonts for machines where it is not installed. TrueType (42)
# embedding keeps text selectable/editable in PDF and PostScript output.
FIGURE_RC_PARAMS = {
    "font.family": "serif",
    "font.serif": [
        "Times New Roman", "Times", "Nimbus Roman", "Liberation Serif",
        "FreeSerif", "STIXGeneral", "DejaVu Serif",
    ],
    "mathtext.fontset": "stix",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}


# ---------------------------------------------------------------------
# Output formats
# ---------------------------------------------------------------------

@dataclass(frozen=True)
class FileFormat:
    """One selectable output format."""

    label: str
    extension: str
    savefig_kwargs: Mapping[str, object] = field(default_factory=dict)
    requires: str | None = None  # key into OPTIONAL_DEPENDENCIES


# Keys of IMAGE_FORMATS double as Matplotlib format names (except AVIF,
# which goes through Pillow, see `save_figure`).
IMAGE_FORMATS: dict[str, FileFormat] = {
    "png": FileFormat("PNG", ".png"),
    "jpg": FileFormat("JPEG", ".jpg", {"pil_kwargs": {"quality": 95}}),
    "svg": FileFormat("SVG (vector)", ".svg"),
    "pdf": FileFormat("PDF (vector)", ".pdf"),
    "tiff": FileFormat("TIFF", ".tiff", {"pil_kwargs": {"compression": "tiff_lzw"}}),
    "avif": FileFormat("AVIF", ".avif", requires="avif"),
}

TABLE_FORMATS: dict[str, FileFormat] = {
    "csv": FileFormat("CSV (comma separated)", ".csv"),
    "txt": FileFormat("TXT (tab separated)", ".txt"),
    "xlsx": FileFormat("Excel (.xlsx)", ".xlsx", requires="excel"),
}

AVIF_QUALITY = 90


@functools.cache
def _avif_supported() -> bool:
    """AVIF needs Pillow >= 11.3 or the third-party ``pillow-avif-plugin``."""
    try:
        from PIL import Image, features
    except ImportError:
        return False
    if features.check("avif"):
        return True
    if importlib.util.find_spec("pillow_avif") is None:
        return False
    import pillow_avif  # noqa: F401  (registers the AVIF codec on import)

    Image.init()
    return "AVIF" in Image.SAVE


@functools.cache
def _excel_engine() -> str | None:
    """Fastest installed engine for writing .xlsx files, if any."""
    for engine in ("xlsxwriter", "openpyxl"):
        if importlib.util.find_spec(engine) is not None:
            return engine
    return None


OPTIONAL_DEPENDENCIES: dict[str, tuple[Callable[[], bool], str]] = {
    "avif": (_avif_supported, "requires Pillow 11.3+ or: pip install pillow-avif-plugin"),
    "excel": (lambda: _excel_engine() is not None, "requires: pip install openpyxl"),
}


def unavailable_reason(file_format: FileFormat) -> str | None:
    """Why `file_format` cannot be used on this machine, or None if it can."""
    if file_format.requires is None:
        return None
    is_available, hint = OPTIONAL_DEPENDENCIES[file_format.requires]
    return None if is_available() else hint


# ---------------------------------------------------------------------
# Normalization methods
# ---------------------------------------------------------------------
# Each entry describes one way of turning a raw radiance window into the
# values that actually get plotted: a checkbox label for the UI, the
# output subfolder, the y-axis label, and whether it needs the internal
# standard wavelength to be set. Order here also controls display order.

NORMALIZATION_METHODS: dict[str, dict] = {
    "raw": {
        "label": "Raw (not normalized)",
        "folder": "Spectrum_plots_RAW",
        "ylabel": "Radiance [W/(sr\u00b7m\u00b2\u00b7nm)]",
        "needs_internal_standard": False,
    },
    "auc": {
        "label": "AUC normalized",
        "folder": "Spectrum_plots_AUC",
        "ylabel": "AUC-normalized radiance",
        "needs_internal_standard": False,
    },
    "minmax": {
        "label": "Min-Max normalized",
        "folder": "Spectrum_plots_MinMax",
        "ylabel": "Min-Max normalized radiance",
        "needs_internal_standard": False,
    },
    "vector": {
        "label": "Vector normalized (L2)",
        "folder": "Spectrum_plots_Vector",
        "ylabel": "Vector-normalized radiance",
        "needs_internal_standard": False,
    },
    "meancenter": {
        "label": "Mean-centered",
        "folder": "Spectrum_plots_MeanCentered",
        "ylabel": "Mean-centered radiance",
        "needs_internal_standard": False,
    },
    "exposure": {
        "label": "Integration-time normalized",
        "folder": "Spectrum_plots_IntegrationTime",
        "ylabel": "Radiance per integration time [W/(sr\u00b7m\u00b2\u00b7nm\u00b7ms)]",
        "needs_internal_standard": False,
    },
    "internalstd": {
        "label": "Internal standard normalized",
        "folder": "Spectrum_plots_InternalStd",
        "ylabel": "Internal-standard normalized radiance",
        "needs_internal_standard": True,
    },
}

# numpy 2 renamed `trapz` to `trapezoid`; support both.
_trapezoid = getattr(np, "trapezoid", None) or np.trapz


_INTEGRATION_TIME_PATTERN = re.compile(r"^\s*(\d+(?:\.\d+)?)\s*\*\s*(\d+(?:\.\d+)?)\s*$")


def parse_integration_time_ms(value: object) -> float | None:
    """Total integration time in ms from a ``T_int [ms]`` header value.

    JETI writes ``"<count> * <time per integration>"`` (e.g. ``"14 * 500.000"``,
    i.e. 14 x 500 ms = 7000 ms); a plain number is taken as the total.
    Returns None if the value is missing, unreadable or not positive.
    """
    if isinstance(value, (int, float)):
        total_ms = float(value)
    else:
        match = _INTEGRATION_TIME_PATTERN.match(str(value))
        if match is None:
            return None
        total_ms = float(match.group(1)) * float(match.group(2))
    return total_ms if np.isfinite(total_ms) and total_ms > 0 else None


def compute_normalized_radiance(
    method_key: str,
    wavelength_window: np.ndarray,
    radiance_window: np.ndarray,
    internal_standard_wavelength_nm: float | None = None,
    integration_time_ms: float | None = None,
) -> tuple[np.ndarray | None, str | None]:
    """Apply one normalization method to a (wavelength, radiance) window.

    Returns (normalized_radiance, error_message). On success `error_message`
    is None; on failure (e.g. division by zero) `normalized_radiance` is
    None and `error_message` explains why, so the caller can skip and log it.
    """
    if method_key == "raw":
        return radiance_window, None

    if method_key == "auc":
        area_under_curve = _trapezoid(radiance_window, wavelength_window)
        if area_under_curve == 0 or not np.isfinite(area_under_curve):
            return None, "AUC is zero or non-finite"
        return radiance_window / area_under_curve, None

    if method_key == "minmax":
        range_min = np.min(radiance_window)
        span = np.max(radiance_window) - range_min
        if span == 0 or not np.isfinite(span):
            return None, "min equals max (zero range)"
        return (radiance_window - range_min) / span, None

    if method_key == "vector":
        vector_norm = np.linalg.norm(radiance_window)
        if vector_norm == 0 or not np.isfinite(vector_norm):
            return None, "vector norm is zero or non-finite"
        return radiance_window / vector_norm, None

    if method_key == "meancenter":
        return radiance_window - np.mean(radiance_window), None

    if method_key == "exposure":
        if integration_time_ms is None:
            return None, "no valid integration time found in the file header"
        return radiance_window / integration_time_ms, None

    if method_key == "internalstd":
        if internal_standard_wavelength_nm is None:
            return None, "no internal standard wavelength was set"
        if not (wavelength_window[0] <= internal_standard_wavelength_nm <= wavelength_window[-1]):
            return None, "internal standard wavelength is outside this window"
        reference_value = np.interp(
            internal_standard_wavelength_nm, wavelength_window, radiance_window
        )
        if reference_value == 0 or not np.isfinite(reference_value):
            return None, "internal standard value is zero or non-finite"
        return radiance_window / reference_value, None

    raise ValueError(f"Unknown normalization method: {method_key!r}")


# =====================================================================
# Data model
# =====================================================================

@dataclass
class Spectrum:
    """One observation (one CSV column of one file) with all its data."""

    name: str                       # observation name as written in the file
    source_file: str
    column_index: int               # 1-based position of the column in the file
    metadata: dict[str, object]     # every header field, in file order
    blocks: dict[str, pd.Series]    # spectral blocks, indexed by wavelength

    @property
    def wavelength_nm(self) -> np.ndarray:
        return self.blocks[RADIANCE_BLOCK_KEY].index.to_numpy(dtype=float)

    @property
    def radiance(self) -> np.ndarray:
        return self.blocks[RADIANCE_BLOCK_KEY].to_numpy(dtype=float)

    @property
    def integration_time_ms(self) -> float | None:
        return parse_integration_time_ms(self.metadata.get(INTEGRATION_TIME_LABEL))

    def spectral_table(self) -> pd.DataFrame:
        """All spectral blocks side by side, aligned on wavelength."""
        return (
            pd.concat(self.blocks.values(), axis=1)
            .sort_index()
            .rename_axis(WAVELENGTH_LABEL)
            .reset_index()
        )


@dataclass
class GroupItem:
    """One row of a plot group: which observation, what label/colour to use."""

    column_name: str
    display_name: str
    color: str


@dataclass
class PlotGroup:
    """A named set of observations that will be drawn on the same figure."""

    title: str
    items: list[GroupItem] = field(default_factory=list)
    show_title_on_figure: bool = True  # False: the title only names the files


# =====================================================================
# CSV parsing
# =====================================================================

def _read_text(filepath: Path) -> str:
    """Decode as UTF-8 when valid, else Latin-1 (what JETI software writes)."""
    raw = filepath.read_bytes()
    try:
        return raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        return raw.decode("latin1")


def _is_blank(row: Sequence[str]) -> bool:
    return not any(cell.strip() for cell in row)


def _is_block_header(row: Sequence[str]) -> bool:
    return bool(row) and row[0].strip() == WAVELENGTH_LABEL


def _decode_decimal_fields(fields: Sequence[str], count: int) -> list[str] | None:
    """Rebuild `count` numbers from a row's fields, or None if the row
    matches neither layout.

    ``count`` fields  -> one field per value (fixed layout).
    ``2 * count``     -> (integer part, fractional part) pairs, so that
                         ["1234", "56"] becomes "1234.56" (legacy layout).
    """
    if len(fields) == count:
        return [cell.strip() for cell in fields]
    if len(fields) == 2 * count:
        return [
            f"{fields[i].strip()}.{fields[i + 1].strip()}"
            for i in range(0, 2 * count, 2)
        ]
    return None


def _coerce_scalar(text: str) -> int | float | str:
    """Best-effort typed value: int, else float, else the text unchanged."""
    text = text.strip()
    for cast in (int, float):
        try:
            return cast(text)
        except ValueError:
            continue
    return text


def _parse_header(
    header_rows: Iterable[Sequence[str]],
    spectrum_count: int,
    filename: str,
    warnings: list[str],
) -> list[dict[str, object]]:
    """Read the metadata/scalar rows preceding the spectral blocks.

    Returns one ``{column label: value}`` dict per spectrum. Three kinds of
    rows exist:

    * ``label, v1 ... vN``     -> one value per spectrum,
    * ``key: value``           -> a value shared by every spectrum,
    * ``Section title`` alone  -> prefixes the following rows (until the next
      blank row), e.g. the ``x`` / ``y`` rows under the chromaticity title.
    """
    metadata: list[dict[str, object]] = [{} for _ in range(spectrum_count)]
    section = ""

    for row in header_rows:
        label = row[0].strip() if row else ""
        values = row[1:]

        if _is_blank(row):
            section = ""
        elif label == NAME_LABEL:
            continue  # observation names are handled separately
        elif not label:
            warnings.append(f"{filename}: header row without a label was skipped.")
        elif _is_blank(values):
            key, _, value = label.partition(":")
            if value.strip():
                for record in metadata:
                    record[key.strip()] = _coerce_scalar(value)
            else:
                section = label
        else:
            decoded = _decode_decimal_fields(values, spectrum_count)
            if decoded is None:
                warnings.append(
                    f"{filename}: header row '{label}' has an unexpected number "
                    "of fields and was skipped."
                )
                continue
            column = f"{section} - {label}" if section else label
            for record, text in zip(metadata, decoded):
                record[column] = _coerce_scalar(text)

    return metadata


def _decode_block_row(row: Sequence[str], spectrum_count: int) -> tuple[float, list[float]] | None:
    """Decode ``[wavelength, *values]`` from one spectral row, or None."""
    field_count = len(row)
    if field_count in (1 + spectrum_count, 1 + 2 * spectrum_count):
        wavelength_text, value_fields = row[0], row[1:]
    elif field_count == 2 + 2 * spectrum_count:  # non-integer wavelength, split too
        wavelength_text, value_fields = f"{row[0]}.{row[1]}", row[2:]
    else:
        return None

    value_texts = _decode_decimal_fields(value_fields, spectrum_count)
    try:
        return float(wavelength_text), [float(text) for text in value_texts or ()]
    except ValueError:
        return None


def _parse_block(
    header: Sequence[str],
    data_rows: Sequence[Sequence[str]],
    spectrum_count: int,
    filename: str,
    warnings: list[str],
) -> tuple[str, list[pd.Series]] | None:
    """Turn one ``Wavelength [nm],<label>,...`` block into one Series per spectrum."""
    block_key = header[1].strip() if len(header) > 1 else ""
    if not block_key:
        warnings.append(f"{filename}: a spectral block without a label was skipped.")
        return None

    wavelengths: list[float] = []
    values: list[list[float]] = []
    unreadable_rows = 0
    for row in data_rows:
        decoded = _decode_block_row(row, spectrum_count)
        if decoded is None:
            unreadable_rows += 1
            continue
        wavelengths.append(decoded[0])
        values.append(decoded[1])

    if unreadable_rows:
        warnings.append(
            f"{filename}: {unreadable_rows} unreadable row(s) skipped in block '{block_key}'."
        )
    if not wavelengths:
        warnings.append(f"{filename}: block '{block_key}' contains no readable data.")
        return None

    # Sorted, one row per wavelength (first occurrence wins).
    unique_wavelengths, first_indices = np.unique(np.asarray(wavelengths), return_index=True)
    value_matrix = np.asarray(values)[first_indices]

    series_per_spectrum: list[pd.Series] = []
    for column in range(spectrum_count):
        column_values = value_matrix[:, column]
        finite = np.isfinite(column_values)
        series_per_spectrum.append(
            pd.Series(
                column_values[finite],
                index=pd.Index(unique_wavelengths[finite], name=WAVELENGTH_LABEL),
                name=block_key,
            )
        )
    return block_key, series_per_spectrum


def _iter_blocks(rows: Sequence[Sequence[str]]) -> Iterator[tuple[Sequence[str], list[Sequence[str]]]]:
    """Yield (header_row, data_rows) for every spectral block. A block ends
    at the next blank row or at the next block header."""
    position = 0
    while position < len(rows):
        header = rows[position]
        position += 1
        if not _is_block_header(header):
            continue
        start = position
        while position < len(rows) and not _is_blank(rows[position]) and not _is_block_header(rows[position]):
            position += 1
        yield header, list(rows[start:position])


def parse_spectrum_file(filepath: Path) -> tuple[list[Spectrum], list[str]]:
    """Parse one JETI export into one `Spectrum` per data column.

    Returns (spectra, warnings). Problems never raise: unreadable files or
    rows are reported in `warnings` so one bad file cannot block the rest.
    """
    filename = filepath.name
    warnings: list[str] = []

    try:
        rows = list(csv.reader(_read_text(filepath).splitlines()))
    except OSError as exc:
        return [], [f"{filename}: cannot read file ({exc})."]

    name_row = next((row for row in rows if row and row[0].strip() == NAME_LABEL), None)
    if name_row is None:
        return [], [f"{filename}: could not find the '{NAME_LABEL}' row."]
    names = [cell.strip() for cell in name_row[1:]]
    while names and not names[-1]:
        names.pop()
    if not names:
        return [], [f"{filename}: the '{NAME_LABEL}' row holds no observation."]
    spectrum_count = len(names)

    first_block = next((i for i, row in enumerate(rows) if _is_block_header(row)), None)
    if first_block is None:
        return [], [f"{filename}: could not find any spectral data block."]

    metadata = _parse_header(rows[:first_block], spectrum_count, filename, warnings)

    blocks: dict[str, list[pd.Series]] = {}
    for header, data_rows in _iter_blocks(rows[first_block:]):
        parsed = _parse_block(header, data_rows, spectrum_count, filename, warnings)
        if parsed is None:
            continue
        block_key, series_per_spectrum = parsed
        if block_key in blocks:
            warnings.append(f"{filename}: block '{block_key}' appears twice; only the first was kept.")
            continue
        blocks[block_key] = series_per_spectrum

    if RADIANCE_BLOCK_KEY not in blocks:
        warnings.append(f"{filename}: could not find the radiance spectrum block.")
        return [], warnings

    spectra = [
        Spectrum(
            name=name,
            source_file=filename,
            column_index=column + 1,
            metadata=metadata[column],
            blocks={key: series[column] for key, series in blocks.items()},
        )
        for column, name in enumerate(names)
    ]
    return spectra, warnings


# =====================================================================
# Directory-level loading
# =====================================================================

def load_all_spectra(data_dir: Path) -> tuple[list[Spectrum], list[str]]:
    """Load every CSV of `data_dir` into a flat list of `Spectrum` objects.

    Returns (spectra, warnings); `spectra` is empty if nothing could be read.
    Our own combined export is skipped so re-running the tool on the same
    folder never re-imports it.
    """
    csv_paths = sorted(
        path for path in data_dir.iterdir()
        if path.is_file() and path.suffix.lower() == ".csv" and path.stem != COMBINED_FILENAME_STEM
    )
    if not csv_paths:
        return [], ["No CSV files were found in the selected directory."]

    spectra: list[Spectrum] = []
    warnings: list[str] = []
    for path in csv_paths:
        file_spectra, file_warnings = parse_spectrum_file(path)
        spectra.extend(file_spectra)
        warnings.extend(file_warnings)

    if not spectra:
        warnings.append("No valid spectra were found.")
    return spectra, warnings


def group_spectra_by_name(spectra: Iterable[Spectrum]) -> dict[str, list[Spectrum]]:
    """Index spectra by observation name, keeping file order within a name."""
    by_name: dict[str, list[Spectrum]] = {}
    for spectrum in spectra:
        by_name.setdefault(spectrum.name, []).append(spectrum)
    return by_name


def slugify_for_filename(text: str) -> str:
    """Turn an arbitrary plot title into a filesystem-safe filename fragment."""
    slug = re.sub(r"[^\w-]+", "_", text.strip())
    return re.sub(r"_+", "_", slug).strip("_") or "plot"


# =====================================================================
# Combined table export
# =====================================================================

def build_combined_table(spectra: Sequence[Spectrum]) -> pd.DataFrame:
    """One row per (spectrum, wavelength); every piece of information of the
    original files sits in its own column:

    ``Source file | Column in file | Name | <every header field> |
    Wavelength [nm] | <one column per spectral block>``

    Header fields (date, integration time, luminance, CCT, chromaticity, ...)
    repeat on each row of their spectrum, which keeps the table flat and
    directly filterable. Blocks with different wavelength ranges (e.g.
    radiance and raw counts) are aligned on wavelength; missing cells stay
    empty.
    """
    metadata_table = pd.DataFrame([
        {
            "Source file": spectrum.source_file,
            "Column in file": spectrum.column_index,
            NAME_LABEL: spectrum.name,
            **spectrum.metadata,
        }
        for spectrum in spectra
    ])
    spectral_tables = [spectrum.spectral_table() for spectrum in spectra]

    row_counts = [len(table) for table in spectral_tables]
    repeated_metadata = metadata_table.iloc[
        np.repeat(np.arange(len(spectra)), row_counts)
    ].reset_index(drop=True)
    return pd.concat(
        [repeated_metadata, pd.concat(spectral_tables, ignore_index=True)], axis=1
    )


def _write_table(table: pd.DataFrame, path: Path, format_key: str) -> None:
    # utf-8-sig: Excel then displays characters such as the degree sign correctly.
    if format_key == "csv":
        table.to_csv(path, index=False, encoding="utf-8-sig")
    elif format_key == "txt":
        table.to_csv(path, sep="\t", index=False, encoding="utf-8-sig")
    elif format_key == "xlsx":
        if len(table) + 1 > EXCEL_MAX_ROWS:
            raise ValueError(
                f"{len(table):,} rows exceed Excel's limit of {EXCEL_MAX_ROWS - 1:,}; use CSV or TXT"
            )
        table.to_excel(
            path, index=False, sheet_name="Spectra",
            engine=_excel_engine(), freeze_panes=(1, 0),
        )
    else:
        raise ValueError(f"Unknown table format: {format_key!r}")


def export_combined_table(
    spectra: Sequence[Spectrum], output_dir: Path, format_keys: Sequence[str]
) -> tuple[list[Path], list[str]]:
    """Write the combined table in every requested format.

    Returns (written_paths, errors). Failures are reported, never raised, so
    one failing format does not prevent the others from being written.
    """
    if not spectra:
        return [], ["No spectra available to export."]

    try:
        table = build_combined_table(spectra)
    except Exception as exc:  # pandas alignment/type problems
        return [], [f"Could not assemble the combined table: {exc}"]

    written: list[Path] = []
    errors: list[str] = []
    for key in format_keys:
        file_format = TABLE_FORMATS[key]
        path = output_dir / f"{COMBINED_FILENAME_STEM}{file_format.extension}"
        try:
            _write_table(table, path, key)
        except PermissionError:
            errors.append(f"{path.name}: permission denied (is the file open in another program?)")
        except Exception as exc:
            errors.append(f"{path.name}: {exc}")
        else:
            written.append(path)
    return written, errors


# =====================================================================
# Figures
# =====================================================================

def build_group_figure(
    observation_pairs: Sequence[tuple[Spectrum, GroupItem]],
    group: PlotGroup,
    method_key: str,
    wavelength_range_nm: tuple[float, float],
    internal_standard_wavelength_nm: float | None,
) -> Figure | None:
    """Draw one group with one normalization method.

    Uses Matplotlib's object-oriented API (no pyplot), so no global state
    or hidden window is created next to the Tk application. Returns None
    when nothing could be drawn. Call inside ``mpl.rc_context(FIGURE_RC_PARAMS)``.
    """
    wavelength_min_nm, wavelength_max_nm = wavelength_range_nm
    method_info = NORMALIZATION_METHODS[method_key]

    figure = Figure(figsize=FIGURE_SIZE_INCHES)
    axes = figure.add_subplot()

    labelled_items: set[str] = set()
    drawn_count = 0
    for spectrum, item in observation_pairs:
        wavelengths = spectrum.wavelength_nm
        in_range = (wavelengths >= wavelength_min_nm) & (wavelengths <= wavelength_max_nm)
        wavelength_window = wavelengths[in_range]
        if len(wavelength_window) < 2:
            log.warning(
                f"Skipped {item.display_name} ({spectrum.source_file}): fewer than 2 points "
                "inside the chosen wavelength range."
            )
            continue

        normalized_radiance, error_message = compute_normalized_radiance(
            method_key,
            wavelength_window,
            spectrum.radiance[in_range],
            internal_standard_wavelength_nm=internal_standard_wavelength_nm,
            integration_time_ms=spectrum.integration_time_ms,
        )
        if error_message is not None:
            log.warning(
                f"Skipped {item.display_name} ({spectrum.source_file}) "
                f"for {method_info['label']}: {error_message}"
            )
            continue

        # Several spectra can share one observation name: label it once.
        first_of_item = item.column_name not in labelled_items
        labelled_items.add(item.column_name)
        axes.plot(
            wavelength_window, normalized_radiance,
            color=item.color, linewidth=1.6, alpha=0.9,
            label=item.display_name if first_of_item else None,
        )
        drawn_count += 1

    if drawn_count == 0:
        return None

    axes.set_xlabel("Wavelength (nm)", fontsize=12)
    axes.set_ylabel(method_info["ylabel"], fontsize=12)
    if group.show_title_on_figure:
        axes.set_title(group.title, fontsize=14, fontweight="bold")
    axes.set_xlim(wavelength_min_nm, wavelength_max_nm)
    axes.grid(True, alpha=0.25)
    axes.legend(fontsize=9)
    figure.tight_layout()
    return figure


def save_figure(
    figure: Figure, output_path: Path, format_key: str, dpi: int
) -> tuple[bool, str | None]:
    """Save `figure` as `format_key`. Returns (success, error_message)."""
    try:
        if format_key == "avif":
            _save_figure_as_avif(figure, output_path, dpi)
        else:
            figure.savefig(
                output_path, format=format_key, dpi=dpi, bbox_inches="tight",
                **IMAGE_FORMATS[format_key].savefig_kwargs,
            )
    except Exception as exc:
        return False, f"{output_path.name}: {exc}"
    return True, None


def _save_figure_as_avif(figure: Figure, output_path: Path, dpi: int) -> None:
    """Matplotlib cannot write AVIF: render a PNG in memory, encode it with Pillow.
    4:4:4 chroma keeps thin coloured lines crisp."""
    from PIL import Image

    if not _avif_supported():
        raise RuntimeError(OPTIONAL_DEPENDENCIES["avif"][1])

    buffer = io.BytesIO()
    figure.savefig(buffer, format="png", dpi=dpi, bbox_inches="tight")
    buffer.seek(0)
    with Image.open(buffer) as image:
        image.convert("RGB").save(
            output_path, format="AVIF", quality=AVIF_QUALITY, subsampling="4:4:4"
        )


def warn_if_preferred_font_missing() -> None:
    try:
        font_manager.findfont("Times New Roman", fallback_to_default=False)
    except ValueError:
        log.warning(
            "Times New Roman is not installed on this system; figures will use "
            "the closest serif font available."
        )


# =====================================================================
# GUI application
# =====================================================================

class SpectrumPlotterApp:
    """Wizard-style Tkinter UI driving the load -> group -> plot flow."""

    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title(APP_TITLE)
        self.root.geometry("1000x750")
        self.root.minsize(800, 600)

        # Data loaded from disk
        self.data_dir: Path | None = None
        self.all_spectra: list[Spectrum] = []
        self.spectra_by_name: dict[str, list[Spectrum]] = {}
        self.column_names: list[str] = []

        # Wizard state carried between screens
        self.plot_groups: list[PlotGroup] = []
        self.wavelength_min_nm: float = 0.0
        self.wavelength_max_nm: float = 0.0
        self.selected_methods: list[str] = ["raw", "auc"]
        self.normalization_vars: dict[str, tk.BooleanVar] = {}
        self.internal_standard_wavelength_nm: float | None = None
        self.internal_standard_entry: ttk.Entry
        self.selected_image_formats: list[str] = ["png"]
        self.image_format_vars: dict[str, tk.BooleanVar] = {}
        self.image_dpi: int = DEFAULT_DPI
        self.dpi_entry: ttk.Entry

        # "What do you want to do" screen
        self.export_table_requested: bool = False
        self.create_plots_requested: bool = True
        self.selected_table_formats: list[str] = ["csv"]
        self.export_table_var: tk.BooleanVar
        self.create_plots_var: tk.BooleanVar
        self.table_format_vars: dict[str, tk.BooleanVar] = {}
        self.table_format_widgets: dict[str, ttk.Checkbutton] = {}

        # Results shown on the final screen
        self.export_report: tuple[list[Path], list[str]] | None = None
        self.saved_image_count: int = 0
        self.image_save_errors: list[str] = []

        # Transient per-screen widget state
        self.selected_column_names: set[str] = set()
        self.column_buttons_by_name: dict[str, tk.Button] = {}
        self.default_button_bg: str = ""
        self.group_title_entry: ttk.Entry
        self.show_title_var: tk.BooleanVar
        self.customization_name_entries: list[ttk.Entry] = []
        self.customization_color_entries: list[dict] = []

        self.show_directory_screen()

    # -----------------------------------------------------------------
    # Shared helpers
    # -----------------------------------------------------------------

    def clear_window(self) -> None:
        for widget in self.root.winfo_children():
            widget.destroy()

    def _busy(self, is_busy: bool) -> None:
        self.root.config(cursor="watch" if is_busy else "")
        self.root.update()

    def _make_scrollable_frame(self, parent: tk.Widget) -> ttk.Frame:
        """Return a scrollable content frame packed into `parent`."""
        container = ttk.Frame(parent)
        container.pack(fill="both", expand=True)

        canvas = tk.Canvas(container, highlightthickness=0)
        scrollbar = ttk.Scrollbar(container, orient="vertical", command=canvas.yview)
        content_frame = ttk.Frame(canvas)

        content_frame.bind(
            "<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
        )
        canvas.create_window((0, 0), window=content_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)

        canvas.pack(side="left", fill="both", expand=True)
        scrollbar.pack(side="right", fill="y")
        return content_frame

    @staticmethod
    def _add_format_checkbuttons(
        parent: tk.Widget,
        formats: Mapping[str, FileFormat],
        variables: dict[str, tk.BooleanVar],
        selected_keys: Iterable[str],
    ) -> dict[str, ttk.Checkbutton]:
        """One checkbox per format; formats whose dependency is missing are
        shown disabled with the reason next to them."""
        selected = set(selected_keys)
        widgets: dict[str, ttk.Checkbutton] = {}
        for key, file_format in formats.items():
            reason = unavailable_reason(file_format)
            variables[key] = tk.BooleanVar(value=key in selected and reason is None)
            text = file_format.label if reason is None else f"{file_format.label}  ({reason})"
            button = ttk.Checkbutton(parent, text=text, variable=variables[key])
            button.pack(anchor="w", pady=2)
            if reason is not None:
                button.state(["disabled"])
            widgets[key] = button
        return widgets

    @staticmethod
    def _checked_keys(variables: Mapping[str, tk.BooleanVar]) -> list[str]:
        return [key for key, variable in variables.items() if variable.get()]

    # -----------------------------------------------------------------
    # Screen 1: choose directory
    # -----------------------------------------------------------------

    def show_directory_screen(self) -> None:
        self.clear_window()
        frame = ttk.Frame(self.root, padding=30)
        frame.pack(fill="both", expand=True)

        ttk.Label(frame, text="Spectrum Plotter", font=("TkDefaultFont", 20, "bold")).pack(pady=(20, 10))
        ttk.Label(
            frame, text="Enter or paste the folder containing your spectrum CSV files.",
            font=("TkDefaultFont", 12),
        ).pack(pady=(0, 25))

        path_frame = ttk.Frame(frame)
        path_frame.pack(fill="x", padx=50)

        self.path_entry = ttk.Entry(path_frame)
        self.path_entry.pack(side="left", fill="x", expand=True, ipady=6)
        ttk.Button(path_frame, text="Browse...", command=self._browse_directory).pack(side="left", padx=(10, 0))

        ttk.Button(frame, text="Load CSV Files", command=self._load_directory, width=25).pack(pady=30)

    def _browse_directory(self) -> None:
        directory = filedialog.askdirectory(title="Select spectrum data directory")
        if directory:
            self.path_entry.delete(0, tk.END)
            self.path_entry.insert(0, directory)

    def _load_directory(self) -> None:
        raw_path = self.path_entry.get().strip()
        if not raw_path:
            messagebox.showerror("Error", "Please enter or select a data directory.")
            return

        directory = Path(raw_path)
        if not directory.is_dir():
            messagebox.showerror("Error", "The selected directory does not exist.")
            return

        self._busy(True)
        try:
            spectra, warnings = load_all_spectra(directory)
        finally:
            self._busy(False)

        for warning in warnings:
            log.warning(warning)

        if not spectra:
            messagebox.showerror("Nothing to load", "\n".join(warnings[:10]))
            return
        if warnings:
            shown = "\n".join(warnings[:8])
            hidden = f"\n... and {len(warnings) - 8} more (see the console log)." if len(warnings) > 8 else ""
            messagebox.showwarning(
                "Some data could not be read",
                f"{len(spectra)} observations were loaded, but:\n\n{shown}{hidden}",
            )

        self.data_dir = directory
        self.all_spectra = spectra
        self.spectra_by_name = group_spectra_by_name(spectra)
        self.column_names = sorted(self.spectra_by_name, key=str.lower)
        self.plot_groups = []
        self.show_export_mode_screen()

    # -----------------------------------------------------------------
    # Screen 2: choose what to do (export table / plot / both)
    # -----------------------------------------------------------------

    def show_export_mode_screen(self) -> None:
        self.clear_window()
        frame = ttk.Frame(self.root, padding=40)
        frame.pack(fill="both", expand=True)

        ttk.Label(frame, text="What would you like to do?", font=("TkDefaultFont", 18, "bold")).pack(
            pady=(20, 10)
        )
        ttk.Label(
            frame,
            text=(
                f"Loaded {len(self.all_spectra)} observations from "
                f"{len(self.column_names)} distinct name(s)."
            ),
        ).pack(pady=(0, 20))

        self.export_table_var = tk.BooleanVar(value=self.export_table_requested)
        self.create_plots_var = tk.BooleanVar(value=self.create_plots_requested)

        ttk.Checkbutton(
            frame,
            text="Save all files into one combined, decimal-corrected table (all data and metadata)",
            variable=self.export_table_var,
            command=self._update_table_format_state,
        ).pack(anchor="w", pady=(6, 2), padx=60)

        format_frame = ttk.Frame(frame)
        format_frame.pack(anchor="w", padx=100, pady=(0, 6))
        ttk.Label(format_frame, text="Save as:").pack(anchor="w")
        self.table_format_vars = {}
        self.table_format_widgets = self._add_format_checkbuttons(
            format_frame, TABLE_FORMATS, self.table_format_vars, self.selected_table_formats
        )
        self._update_table_format_state()

        ttk.Checkbutton(
            frame, text="Create plots", variable=self.create_plots_var
        ).pack(anchor="w", pady=6, padx=60)

        ttk.Label(
            frame,
            text="You can select either option, or both.",
            font=("TkDefaultFont", 8),
        ).pack(pady=(15, 0))

        button_frame = ttk.Frame(frame)
        button_frame.pack(pady=30)
        ttk.Button(button_frame, text="Back", command=self.show_directory_screen).pack(side="left", padx=10)
        ttk.Button(button_frame, text="Continue", command=self._confirm_export_mode).pack(side="left", padx=10)

    def _update_table_format_state(self) -> None:
        """Format checkboxes are only active while table export is ticked."""
        state = "!disabled" if self.export_table_var.get() else "disabled"
        for key, widget in self.table_format_widgets.items():
            if unavailable_reason(TABLE_FORMATS[key]) is None:
                widget.state([state])

    def _confirm_export_mode(self) -> None:
        export_table = self.export_table_var.get()
        create_plots = self.create_plots_var.get()
        table_formats = self._checked_keys(self.table_format_vars)

        if not export_table and not create_plots:
            messagebox.showwarning("Nothing selected", "Please select at least one of the two options.")
            return
        if export_table and not table_formats:
            messagebox.showwarning("No file format", "Please select at least one format for the combined table.")
            return

        self.export_table_requested = export_table
        self.create_plots_requested = create_plots
        self.selected_table_formats = table_formats or self.selected_table_formats
        self.export_report = None

        if export_table:
            self._busy(True)
            try:
                self.export_report = export_combined_table(
                    self.all_spectra, self.data_dir, table_formats
                )
            finally:
                self._busy(False)
            for error in self.export_report[1]:
                log.warning(f"Table export failed: {error}")

        if create_plots:
            self.plot_groups = []
            self.show_column_selection()
        else:
            self.show_finished_screen()

    # -----------------------------------------------------------------
    # Screen 3: pick observations for a group
    # -----------------------------------------------------------------

    def show_column_selection(self) -> None:
        self.clear_window()
        self.selected_column_names = set()

        outer = ttk.Frame(self.root, padding=20)
        outer.pack(fill="both", expand=True)

        ttk.Label(
            outer, text=f"Select observations for Plot {len(self.plot_groups) + 1}",
            font=("TkDefaultFont", 18, "bold"),
        ).pack(pady=(5, 5))
        ttk.Label(outer, text="Click observations to select them. Click again to deselect.").pack(pady=(0, 15))

        self.selection_count_label = ttk.Label(outer, text="Selected: 0")
        self.selection_count_label.pack(pady=(0, 10))

        content_frame = self._make_scrollable_frame(outer)
        self.column_buttons_by_name = {}
        for index, name in enumerate(self.column_names):
            button = tk.Button(
                content_frame, text=name, width=30, height=2, wraplength=220,
                relief="raised", bd=2, command=lambda n=name: self._toggle_column(n),
            )
            button.grid(row=index // BUTTON_COLUMNS, column=index % BUTTON_COLUMNS, padx=8, pady=6, sticky="ew")
            self.column_buttons_by_name[name] = button
            self.default_button_bg = button.cget("bg")  # platform-neutral "unselected" colour

        bottom_bar = ttk.Frame(outer)
        bottom_bar.pack(fill="x", pady=(15, 0))
        ttk.Button(bottom_bar, text="Back", command=self._go_back_from_selection).pack(side="left")
        ttk.Button(bottom_bar, text="CONFIRM SELECTION", command=self._confirm_selection).pack(side="right")

    def _toggle_column(self, name: str) -> None:
        button = self.column_buttons_by_name[name]
        if name in self.selected_column_names:
            self.selected_column_names.remove(name)
            button.config(relief="raised", bg=self.default_button_bg)
        else:
            self.selected_column_names.add(name)
            button.config(relief="sunken", bg="#b8d7e5")
        self.selection_count_label.config(text=f"Selected: {len(self.selected_column_names)}")

    def _confirm_selection(self) -> None:
        if not self.selected_column_names:
            messagebox.showwarning("No selection", "Please select at least one observation.")
            return
        self.show_customization_screen(sorted(self.selected_column_names, key=str.lower))

    def _go_back_from_selection(self) -> None:
        if self.plot_groups:
            self.plot_groups.pop()
            if self.plot_groups:
                self._ask_another_group()
                return
        self.show_export_mode_screen()

    # -----------------------------------------------------------------
    # Screen 4: name the group, then name & colour each observation in it
    # -----------------------------------------------------------------

    def show_customization_screen(self, selected_names: list[str]) -> None:
        self.clear_window()
        ttk.Label(self.root, text="Customize this plot group", font=("TkDefaultFont", 18, "bold")).pack(pady=(20, 5))
        ttk.Label(
            self.root,
            text="Give this plot its own title, then choose the name and colour for each observation.",
        ).pack(pady=(0, 10))

        title_block = ttk.Frame(self.root, padding=(30, 0))
        title_block.pack(fill="x", pady=(0, 15))
        title_row = ttk.Frame(title_block)
        title_row.pack(fill="x")
        ttk.Label(title_row, text="Plot title:", font=("TkDefaultFont", 10, "bold")).pack(side="left")
        self.group_title_entry = ttk.Entry(title_row, width=45)
        self.group_title_entry.pack(side="left", padx=10, ipady=3)

        self.show_title_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(
            title_block,
            text="Also display the title on the figure (unticked: the title only appears in the file name)",
            variable=self.show_title_var,
        ).pack(anchor="w", pady=(8, 0))

        outer = ttk.Frame(self.root)
        outer.pack(fill="both", expand=True, padx=30)
        content_frame = self._make_scrollable_frame(outer)

        self.customization_name_entries = []
        self.customization_color_entries = []

        for index, column_name in enumerate(selected_names):
            row = tk.Frame(content_frame, bd=1, relief="groove", padx=10, pady=10)
            row.pack(fill="x", pady=5)

            tk.Label(
                row, text=column_name, width=28, anchor="w", font=("TkDefaultFont", 10, "bold"),
            ).pack(side="left")

            display_name_entry = ttk.Entry(row, width=28)
            display_name_entry.insert(0, column_name)
            display_name_entry.pack(side="left", padx=10)

            default_color = COLOR_PALETTE[index % len(COLOR_PALETTE)]
            color_button = tk.Button(
                row, text="Choose colour", bg=default_color, width=15,
                command=lambda i=index: self._choose_color(i),
            )
            color_button.pack(side="left", padx=10)
            color_hex_label = tk.Label(row, text=default_color, width=10)
            color_hex_label.pack(side="left")

            self.customization_name_entries.append(display_name_entry)
            self.customization_color_entries.append({
                "column_name": column_name,
                "color": default_color,
                "button": color_button,
                "label": color_hex_label,
            })

        bottom_bar = ttk.Frame(self.root)
        bottom_bar.pack(pady=15)
        ttk.Button(bottom_bar, text="Back", command=self.show_column_selection).pack(side="left", padx=10)
        ttk.Button(bottom_bar, text="Confirm Names & Colours", command=self._confirm_customization).pack(side="left", padx=10)

    def _choose_color(self, index: int) -> None:
        entry = self.customization_color_entries[index]
        chosen = colorchooser.askcolor(color=entry["color"], title="Choose spectrum colour")
        if chosen[1] is None:
            return
        entry["color"] = chosen[1]
        entry["button"].config(bg=chosen[1])
        entry["label"].config(text=chosen[1])

    def _confirm_customization(self) -> None:
        group_title = self.group_title_entry.get().strip()
        if not group_title:
            messagebox.showwarning("Missing title", "Please enter a title for this plot.")
            return

        group_items: list[GroupItem] = []
        for name_entry, color_entry in zip(self.customization_name_entries, self.customization_color_entries):
            display_name = name_entry.get().strip()
            if not display_name:
                messagebox.showwarning(
                    "Missing display name",
                    f"Please enter a display name for:\n\n{color_entry['column_name']}",
                )
                return
            group_items.append(
                GroupItem(color_entry["column_name"], display_name, color_entry["color"])
            )

        self.plot_groups.append(
            PlotGroup(title=group_title, items=group_items, show_title_on_figure=self.show_title_var.get())
        )
        self._ask_another_group()

    def _ask_another_group(self) -> None:
        if messagebox.askyesno("Another group?", "Do you want to create another group of observations?"):
            self.show_column_selection()
        else:
            self.show_wavelength_range_screen()

    # -----------------------------------------------------------------
    # Screen 5: wavelength range (shared by every group)
    # -----------------------------------------------------------------

    def show_wavelength_range_screen(self) -> None:
        self.clear_window()
        frame = ttk.Frame(self.root, padding=40)
        frame.pack(fill="both", expand=True)

        ttk.Label(frame, text="Wavelength range", font=("TkDefaultFont", 18, "bold")).pack(pady=(20, 30))

        ttk.Label(frame, text="Lowest wavelength (nm):").pack()
        self.min_wavelength_entry = ttk.Entry(frame, width=25)
        self.min_wavelength_entry.pack(pady=(5, 25), ipady=5)

        ttk.Label(frame, text="Highest wavelength (nm):").pack()
        self.max_wavelength_entry = ttk.Entry(frame, width=25)
        self.max_wavelength_entry.pack(pady=(5, 30), ipady=5)

        if self.wavelength_max_nm > self.wavelength_min_nm:  # restore after "Back"
            self.min_wavelength_entry.insert(0, f"{self.wavelength_min_nm:g}")
            self.max_wavelength_entry.insert(0, f"{self.wavelength_max_nm:g}")

        button_frame = ttk.Frame(frame)
        button_frame.pack()
        ttk.Button(button_frame, text="Back", command=self.show_column_selection).pack(side="left", padx=10)
        ttk.Button(button_frame, text="Continue", command=self._validate_wavelength_range).pack(side="left", padx=10)

    def _validate_wavelength_range(self) -> None:
        try:
            wavelength_min_nm = float(self.min_wavelength_entry.get().strip().replace(",", "."))
            wavelength_max_nm = float(self.max_wavelength_entry.get().strip().replace(",", "."))
        except ValueError:
            messagebox.showerror("Invalid wavelength", "Please enter numeric wavelength values.")
            return

        if wavelength_min_nm >= wavelength_max_nm:
            messagebox.showerror("Invalid range", "The lowest wavelength must be smaller than the highest wavelength.")
            return

        self.wavelength_min_nm = wavelength_min_nm
        self.wavelength_max_nm = wavelength_max_nm
        self.show_normalization_screen()

    # -----------------------------------------------------------------
    # Screen 6: choose plot types (raw / normalizations)
    # -----------------------------------------------------------------

    def show_normalization_screen(self) -> None:
        self.clear_window()
        frame = ttk.Frame(self.root, padding=40)
        frame.pack(fill="both", expand=True)

        ttk.Label(frame, text="Plot type", font=("TkDefaultFont", 18, "bold")).pack(pady=(20, 10))
        ttk.Label(
            frame, text="Which plots would you like to create? Select any combination."
        ).pack(pady=(0, 15))

        checkbox_frame = ttk.Frame(frame)
        checkbox_frame.pack()

        self.normalization_vars = {}
        for method_key, method_info in NORMALIZATION_METHODS.items():
            variable = tk.BooleanVar(value=method_key in self.selected_methods)
            self.normalization_vars[method_key] = variable
            ttk.Checkbutton(checkbox_frame, text=method_info["label"], variable=variable).pack(
                anchor="w", pady=3
            )

        internal_std_frame = ttk.Frame(frame)
        internal_std_frame.pack(pady=(15, 0))
        ttk.Label(internal_std_frame, text="Internal standard wavelength (nm):").pack(side="left")
        self.internal_standard_entry = ttk.Entry(internal_std_frame, width=12)
        self.internal_standard_entry.pack(side="left", padx=(10, 0), ipady=3)
        if self.internal_standard_wavelength_nm is not None:
            self.internal_standard_entry.insert(0, f"{self.internal_standard_wavelength_nm:g}")
        ttk.Label(
            frame,
            text="(only needed if 'Internal standard normalized' is checked above)",
            font=("TkDefaultFont", 8),
        ).pack(pady=(2, 0))

        button_frame = ttk.Frame(frame)
        button_frame.pack(pady=25)
        ttk.Button(button_frame, text="Back", command=self.show_wavelength_range_screen).pack(side="left", padx=10)
        ttk.Button(button_frame, text="Continue", command=self._validate_plot_types).pack(side="left", padx=10)

    def _validate_plot_types(self) -> None:
        selected_methods = self._checked_keys(self.normalization_vars)
        if not selected_methods:
            messagebox.showwarning("No plot type selected", "Please select at least one plot type.")
            return

        if any(NORMALIZATION_METHODS[key]["needs_internal_standard"] for key in selected_methods):
            raw_value = self.internal_standard_entry.get().strip().replace(",", ".")
            try:
                internal_standard_wavelength_nm = float(raw_value)
            except ValueError:
                messagebox.showerror(
                    "Invalid internal standard wavelength",
                    "Please enter a numeric internal standard wavelength.",
                )
                return
            if not (self.wavelength_min_nm <= internal_standard_wavelength_nm <= self.wavelength_max_nm):
                messagebox.showerror(
                    "Invalid internal standard wavelength",
                    "The internal standard wavelength must fall inside the chosen wavelength range "
                    f"({self.wavelength_min_nm:g} \u2013 {self.wavelength_max_nm:g} nm).",
                )
                return
            self.internal_standard_wavelength_nm = internal_standard_wavelength_nm

        self.selected_methods = selected_methods
        self.show_output_options_screen()

    # -----------------------------------------------------------------
    # Screen 7: image formats and resolution
    # -----------------------------------------------------------------

    def show_output_options_screen(self) -> None:
        self.clear_window()
        frame = ttk.Frame(self.root, padding=40)
        frame.pack(fill="both", expand=True)

        ttk.Label(frame, text="Save options", font=("TkDefaultFont", 18, "bold")).pack(pady=(10, 10))
        group_titles = ", ".join(group.title for group in self.plot_groups)
        ttk.Label(
            frame,
            text=(
                f"Groups: {len(self.plot_groups)} ({group_titles})\n"
                f"Wavelength range: {self.wavelength_min_nm:g} \u2013 {self.wavelength_max_nm:g} nm"
            ),
            justify="center",
            wraplength=800,
        ).pack(pady=(0, 15))

        ttk.Label(frame, text="Image formats (select any combination):").pack(anchor="w", padx=200)
        format_frame = ttk.Frame(frame)
        format_frame.pack(anchor="w", padx=220, pady=(4, 10))
        self.image_format_vars = {}
        self._add_format_checkbuttons(
            format_frame, IMAGE_FORMATS, self.image_format_vars, self.selected_image_formats
        )

        dpi_frame = ttk.Frame(frame)
        dpi_frame.pack(pady=(5, 0))
        ttk.Label(dpi_frame, text="Resolution (DPI):").pack(side="left")
        self.dpi_entry = ttk.Entry(dpi_frame, width=8)
        self.dpi_entry.insert(0, str(self.image_dpi))
        self.dpi_entry.pack(side="left", padx=(10, 0), ipady=3)
        ttk.Label(
            frame,
            text=(
                f"Allowed range {MIN_DPI}\u2013{MAX_DPI}. Applies to PNG, JPEG, TIFF and AVIF; "
                "SVG and PDF are vector formats."
            ),
            font=("TkDefaultFont", 8),
        ).pack(pady=(2, 0))

        button_frame = ttk.Frame(frame)
        button_frame.pack(pady=25)
        ttk.Button(button_frame, text="Back", command=self.show_normalization_screen).pack(side="left", padx=10)
        ttk.Button(button_frame, text="GENERATE PLOTS", command=self._validate_and_generate_plots).pack(
            side="left", padx=10
        )

    def _validate_and_generate_plots(self) -> None:
        image_formats = self._checked_keys(self.image_format_vars)
        if not image_formats:
            messagebox.showwarning("No image format", "Please select at least one image format.")
            return

        try:
            dpi = int(self.dpi_entry.get().strip())
        except ValueError:
            dpi = 0
        if not MIN_DPI <= dpi <= MAX_DPI:
            messagebox.showerror(
                "Invalid resolution", f"Please enter a whole number of DPI between {MIN_DPI} and {MAX_DPI}."
            )
            return

        self.selected_image_formats = image_formats
        self.image_dpi = dpi
        self._generate_plots()

    # -----------------------------------------------------------------
    # Plot generation
    # -----------------------------------------------------------------

    def _resolve_group_items(self, group: PlotGroup) -> list[tuple[Spectrum, GroupItem]]:
        """Expand a group's selections into (spectrum, item) pairs. Every
        spectrum sharing a selected name is included: nothing is filtered out."""
        return [
            (spectrum, item)
            for item in group.items
            for spectrum in self.spectra_by_name.get(item.column_name, ())
        ]

    def _generate_plots(self) -> None:
        output_dirs = {
            method_key: self.data_dir / NORMALIZATION_METHODS[method_key]["folder"]
            for method_key in self.selected_methods
        }
        try:
            for output_dir in output_dirs.values():
                output_dir.mkdir(exist_ok=True)
        except OSError as exc:
            messagebox.showerror("Cannot create output folder", str(exc))
            return

        self.saved_image_count = 0
        self.image_save_errors = []
        generated_group_count = 0

        self._busy(True)
        try:
            for group_number, group in enumerate(self.plot_groups, start=1):
                observation_pairs = self._resolve_group_items(group)
                if not observation_pairs:
                    log.warning(f"Plot group {group_number} ('{group.title}') contains no valid spectra.")
                    continue

                log.info(f"Generating Plot {group_number} ('{group.title}', {len(observation_pairs)} observations)")
                for method_key in self.selected_methods:
                    saved_count, errors = self._create_plot(
                        observation_pairs, group, output_dirs[method_key], group_number, method_key
                    )
                    self.saved_image_count += saved_count
                    self.image_save_errors.extend(errors)
                generated_group_count += 1
        finally:
            self._busy(False)

        for error in self.image_save_errors:
            log.warning(f"Image export failed: {error}")
        self.show_finished_screen(generated_group_count)

    def _create_plot(
        self,
        observation_pairs: list[tuple[Spectrum, GroupItem]],
        group: PlotGroup,
        output_dir: Path,
        group_number: int,
        method_key: str,
    ) -> tuple[int, list[str]]:
        """Draw one figure and save it in every selected format.

        Returns (files_saved, error_messages).
        """
        method_label = NORMALIZATION_METHODS[method_key]["label"]
        file_stem = (
            f"Plot_{group_number}_{slugify_for_filename(group.title)}_"
            f"{self.wavelength_min_nm:g}-{self.wavelength_max_nm:g}nm"
        )

        with mpl.rc_context(FIGURE_RC_PARAMS):
            figure = build_group_figure(
                observation_pairs,
                group,
                method_key,
                (self.wavelength_min_nm, self.wavelength_max_nm),
                self.internal_standard_wavelength_nm,
            )
            if figure is None:
                log.warning(f"Plot {group_number} ('{group.title}') has nothing to draw for {method_label}.")
                return 0, []

            saved_count = 0
            errors: list[str] = []
            for format_key in self.selected_image_formats:
                output_path = output_dir / f"{file_stem}{IMAGE_FORMATS[format_key].extension}"
                success, error = save_figure(figure, output_path, format_key, self.image_dpi)
                if success:
                    saved_count += 1
                    log.info(f"Saved {method_label}: {output_path}")
                else:
                    errors.append(error or f"{output_path.name}: unknown error")
        return saved_count, errors

    # -----------------------------------------------------------------
    # Screen 8: done
    # -----------------------------------------------------------------

    def show_finished_screen(self, generated_group_count: int = 0) -> None:
        self.clear_window()
        frame = ttk.Frame(self.root, padding=40)
        frame.pack(fill="both", expand=True)

        ttk.Label(frame, text="Done!", font=("TkDefaultFont", 18, "bold")).pack(pady=(40, 20))

        status_lines: list[str] = []

        if self.export_table_requested and self.export_report is not None:
            written_paths, export_errors = self.export_report
            if written_paths:
                status_lines.append("Combined table saved to:")
                status_lines.extend(str(path) for path in written_paths)
            status_lines.extend(f"Export problem: {error}" for error in export_errors)
            status_lines.append("")

        if self.create_plots_requested:
            formats = ", ".join(IMAGE_FORMATS[key].label.split(" ")[0] for key in self.selected_image_formats)
            status_lines.append(f"Plot groups generated: {generated_group_count}")
            status_lines.append(
                f"Image files written: {self.saved_image_count} ({formats}; {self.image_dpi} dpi for raster formats)\n"
            )
            status_lines.extend(
                f"{NORMALIZATION_METHODS[key]['label']}: "
                f"{'Created' if key in self.selected_methods else 'Not created'}"
                for key in NORMALIZATION_METHODS
            )
            if self.image_save_errors:
                status_lines.append("")
                status_lines.append(f"{len(self.image_save_errors)} file(s) could not be saved:")
                status_lines.extend(self.image_save_errors[:5])
                if len(self.image_save_errors) > 5:
                    status_lines.append("... (see the console log for the full list)")

        ttk.Label(
            frame,
            text="\n".join(status_lines),
            justify="center",
            font=("TkDefaultFont", 11),
            wraplength=850,
        ).pack(pady=20)

        ttk.Button(frame, text="EXIT", command=self.root.destroy, width=20).pack(pady=30, ipady=5)


# =====================================================================
# Entry point
# =====================================================================

def main() -> None:
    warn_if_preferred_font_missing()
    root = tk.Tk()
    SpectrumPlotterApp(root)
    root.mainloop()


if __name__ == "__main__":
    main()
